# Nagelhus et al. (2023), population decoding of the object

**Paper.** Nagelhus et al. (2023), *Neuron* 111, 2091-2104,
[doi:10.1016/j.neuron.2023.04.008](https://doi.org/10.1016/j.neuron.2023.04.008); claims file written
from the version of record with its STAR Methods and Supplemental information.

**Dataset.** EBRAINS [10.25493/Q0FG-1X8](https://doi.org/10.25493/Q0FG-1X8), version v1, the version
whose DOI the paper cites and the only one published (data check of every file in
`01_single_cells.ipynb`).

The decoding sessions are the 16 sessions of the four population files: 14 tetrode sessions (rats
25987 and 26186, 25 to 39 cells) and the two Neuropixels sessions (135 and 620 cells). This is the
legends' count (16 sessions, 14 tetrode). The Methods' text says 15 tetrode sessions, which the
dataset does not hold, so the claims file's session readings (S-methods, S-legends) give the same
16 sessions. Every decoder runs under the claims file's two speed readings (`speed` a: rate maps and
occupancy from epochs above 2.5 cm/s; b: no speed filter) and the two rate-map readings (`rm`).

| Claim | Tested or blocked |
|---|---|
| fig3b-decode-presence | tested |
| fig3c-bin-size | tested |
| fig4b-distance-decline | tested |
| fig4c-each-session-decreasing | tested |
| fig4e-angle-flat | tested |
| fig4h-population-vector-correlation | tested |
| figs5a-decode-no-from-no-prime | tested (tetrode sessions with the cells of the single-cell file only) |
| figs5b-fixed-distance-bins | tested |
| figs5c-half-trial-control | tested |
| figs5d-control-sessions | blocked: one control session has at least 25 cells (shown as a variant) |
| figs5e-shuffled-labels | tested |
| figs5f-random-partition | tested |
| figs5ghi-fields-and-rates | tested |
| figs6-behaviour-decoders-flat | speed and occupancy tested; head direction not in the dataset |
| figs6hij-behaviour-matched | speed and occupancy tested; head direction not in the dataset |
| figs6k-uniform-prior | tested |
| figs6l-sampling-thresholds | tested |
| tables1-without-bin-1 | tested |
| figs5jk-walls-no-pivot | tested |
| figs5m-without-object-tuned | tested |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import objcode as oc

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", "{:.4g}".format)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)

started = time.time()
RAW_POP = []
for p in oc.POP_FILES:
    RAW_POP += oc.load(p)[0]
CA1, _ = oc.load(oc.SINGLE_FILES["CA1"])
POP, replaced = oc.decoding_sessions(RAW_POP, CA1)
print("Sessions whose tracking is taken from the CA1 file (y equals x in the population file):", replaced)
CON1, _ = oc.load(oc.SINGLE_FILES["CON_CA1"])
print(f"dataset {oc.DATASET_ID} version {oc.DATASET_VERSION}: read in {time.time() - started:.0f} s")
READINGS = [(sp, rm) for sp in "ab" for rm in "ab"]
SESS = [f"{s.rat}#{s.index}" for s in POP]

In [ ]:
rows = []
for s in POP:
    NO, O = s.trials
    rows.append({"session": f"{s.rat}#{s.index}", "technique": s.technique, "cells": s.n_units,
                 "NO s": round(NO.duration), "O s": round(O.duration), "dt ms": round(1000 * NO.dt, 1),
                 "object": f"({O.obj[0]:.1f}, {O.obj[1]:.1f})",
                 "1 s test samples NO": sum(len(p.starts) for p in oc.half_parts(NO, True)),
                 "1 s test samples O": sum(len(p.starts) for p in oc.half_parts(O, True)),
                 "speed>2.5 NO": round(np.mean(NO.speed > 2.5), 2), "speed>2.5 O": round(np.mean(O.speed > 2.5), 2)})
info = pd.DataFrame(rows)
print(info.to_string(index=False))
print("Sessions:", len(POP), "; tetrode", int((info.technique == "tetrodes").sum()), "; cells", info.cells.sum())
LINK = {f"{p.rat}#{p.index}": oc.link_population(p, CA1) for p in RAW_POP}
print("Linked single-cell sessions (all with a No Object' trial):",
      all(LINK[k][0] is not None and LINK[k][0].trial("NOp") is not None for k in LINK))

In the two Neuropixels population files the tracking y is a copy of x (notebook 01); their No
Object and Object trials are taken from the CA1 file, which holds the same recordings with the same
timestamps, x and spike trains. The sessions last 913 to 2334 s per trial, not the Methods' 1800 s, so the testing samples per trial
(both iterations) range from about 900 to 2300, not 1800. Every population session is the same
recording as a session of the CA1 single-cell file (notebook 01), which also has its Object Moved
and No Object' trials, but only for the cells the file gives there.

In [ ]:
PARTS = {}
for s in POP:
    for sp in "ab":
        PARTS[s.index, s.rat, sp] = (oc.half_parts(s.trials[0], sp == "a"), oc.half_parts(s.trials[1], sp == "a"))
CACHE = {}


def run(mode="distance", sp="a", rm="a", units=None, sessions=None, **kw):
    """Decoder on every decoding session; returns (sessions, bins) accuracies and the results."""
    out, res = [], []
    for s in (POP if sessions is None else sessions):
        pA, pB = PARTS[s.index, s.rat, sp]
        u = units(s) if callable(units) else units
        r = oc.decode_pair(pA, pB, mode, obj=s.trials[1].obj, rm=rm, units=u, cache=CACHE.setdefault((s.index, s.rat, sp), {}), **kw)
        out.append(r["acc"])
        res.append(r)
    return np.array(out), res


def kw_table(A):
    h, p = oc.kruskal(A.T)
    return h, p


MAIN = {}
for sp, rm in READINGS:
    MAIN[sp, rm] = run("distance", sp, rm, return_samples=True)
print(f"main decoder run under 4 readings ({time.time() - started:.0f} s)")

## fig3b-decode-presence

> A Bayesian decoder using 1 s population spike counts told the NO trial from the O trial above
> chance (50%) in every session.

In [ ]:
rows = []
for sp, rm in READINGS:
    A, res = run("arena", sp, rm, return_samples=True)
    for k, (s, r) in enumerate(zip(POP, res)):
        rng = np.random.default_rng(oc.seed_for("fig3b-decode-presence", sp, rm, k))
        sh = oc.shuffled_accuracy(r["samples"], rng, k=1, n_shuffle=50)
        rows.append({"speed": sp, "rm": rm, "session": SESS[k], "cells": s.n_units, "accuracy": A[k, 0],
                     "shuffle mean": sh.mean(), "shuffle max": sh.max(), "ties": int(r["ties"][0]),
                     "above": A[k, 0] > 0.5 and A[k, 0] > sh.mean()})
fig3b = pd.DataFrame(rows)
print(fig3b.pivot_table(index="session", columns=["speed", "rm"], values="accuracy").to_string())
print("\nShuffle means, range:", fig3b["shuffle mean"].min().round(3), "to", fig3b["shuffle mean"].max().round(3))
print("Sessions above 0.5 and the shuffle mean, per reading:")
print(fig3b.groupby(["speed", "rm"]).above.agg(["sum", "size"]).to_string())

**Result: reproduced.** Under all four readings the whole-arena decoder tells NO from O above 0.5,
and above the mean of 50 label shuffles, in all 16 sessions. Accuracies are 0.61 to 0.77 (table
above), and shuffle means 0.49 to 0.51.

## fig3c-bin-size

> Decoding accuracy grew roughly logarithmically with the testing bin size, rising quickly up to 1 s
> and saturating beyond.

In [ ]:
SIZES = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
rows = []
t0 = time.time()
for sp, rm in READINGS:
    med = []
    for dt in SIZES:
        accs = []
        for s in POP:
            NO, O = s.trials
            pA, pB = oc.half_parts(NO, sp == "a", dt), oc.half_parts(O, sp == "a", dt)
            c = CACHE.setdefault((s.index, s.rat, sp), {})
            # the training maps depend only on the training samples: reuse the 1 s parts' models
            # in a cache local to these parts
            local = {}
            for p_new, p_old in zip(pA + pB, PARTS[s.index, s.rat, sp][0] + PARTS[s.index, s.rat, sp][1]):
                for key in [k for k in c if k[0] == "model" and k[1] == id(p_old)]:
                    local[("model", id(p_new)) + key[2:]] = c[key]
            r = oc.decode_pair(pA, pB, "arena", obj=O.obj, rm=rm, dt=dt, cache=local)
            accs.append(r["acc"][0])
        med.append(np.median(accs))
        rows.append({"speed": sp, "rm": rm, "bin ms": int(dt * 1000), "median": np.median(accs),
                     "min": np.min(accs), "max": np.max(accs)})
    med = np.array(med)
    rises = bool(np.all(np.diff(med[:5]) > 0))
    sat = (med[6] - med[4]) < (med[4] - med[2])
    print(f"speed {sp}, rm {rm}: medians {np.round(med, 3)}; rises 10->1000 ms at each step: {rises}; "
          f"gain 1-10 s {med[6] - med[4]:.3f} < gain 0.1-1 s {med[4] - med[2]:.3f}: {sat}")
fig3c = pd.DataFrame(rows)
print(fig3c.pivot_table(index="bin ms", columns=["speed", "rm"], values="median").to_string())
print(f"({time.time() - t0:.0f} s)")

**Result: reproduced.** Under every reading the median accuracy over sessions rises at each step
from 10 ms (0.51) to 1000 ms (0.69). Its gain from 1 to 10 s (0.031 to 0.045) is smaller than its
gain from 100 ms to 1 s (0.112 to 0.118).

## fig4b-distance-decline

> Decoding accuracy fell gradually with the animal's distance from the object, over 5 equal-sample
> distance bins, with significant steps between bins 1 and 2, 2 and 3, and 4 and 5.

In [ ]:
rows = []
for (sp, rm), (A, res) in MAIN.items():
    h, p = kw_table(A)
    med = np.median(A, 0)
    pairs = [oc.signrank(A[:, i], A[:, i + 1]) for i in range(4)]
    ok_pairs = [pairs[i][1] < 0.05 and med[i] > med[i + 1] for i in (0, 1, 3)]
    rows.append({"speed": sp, "rm": rm, "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                 "monotone": bool(np.all(np.diff(med) < 0)), "1v2 p": pairs[0][1], "2v3 p": pairs[1][1],
                 "3v4 p": pairs[2][1], "4v5 p": pairs[3][1], "pairs ok": sum(ok_pairs),
                 "mean bin1-bin5": np.mean(A[:, 0] - A[:, 4]),
                 "ties": int(sum(r["ties"].sum() for r in res)), "samples": int(sum(r["n"].sum() for r in res))})
fig4b = pd.DataFrame(rows)
print(fig4b.to_string(index=False))
A = MAIN["a", "a"][0]
print("\nPer session (speed a, rm a):")
print(pd.DataFrame(A, index=SESS, columns=[f"bin {i}" for i in range(1, 6)]).to_string())
print("\nVariant: the two Neuropixels sessions with the tracking as deposited in the population files (y = x), speed a, rm a:")
for p in RAW_POP:
    if p.technique == "neuropixels":
        NO, O = p.trials
        r = oc.decode_pair(oc.half_parts(NO, True), oc.half_parts(O, True), "distance", obj=O.obj, rm="a")
        print(f"  {p.rat}: accuracy per bin {np.round(r['acc'], 3)}")
fig, ax = plt.subplots(figsize=(5, 3.5))
for row in A:
    ax.plot(range(1, 6), row, color="0.7", lw=0.8)
ax.plot(range(1, 6), np.median(A, 0), color="k", lw=2)
ax.set_xlabel("distance bin")
ax.set_ylabel("decoding accuracy")
plt.tight_layout()
plt.show()
print("Plotted range:", A.min().round(3), "to", A.max().round(3))

**Result: reproduced.** Under all four readings the Kruskal-Wallis test across the five bins gives
H = 48.4 to 50.9 (paper 44.6), p < 1e-8. The median accuracy falls at every step, from bin 1 (0.79 to
0.80) to bin 5 (0.59 to 0.60). The signed-rank tests of bins 1 vs 2 (p ≤ 0.0012), 2 vs 3 (p ≤ 0.0010)
and 4 vs 5 (p ≤ 0.0008) each give the nearer bin higher. The mean of bin 1 minus bin 5 over sessions
is 0.18 to 0.19 (paper 0.19). With the Neuropixels tracking as deposited (y = x), the two sessions
give a less regular profile (the variant printed above).

## fig4c-each-session-decreasing

> Every session showed decoding accuracy decreasing with distance from the object.

In [ ]:
rows = []
for (sp, rm), (A, _) in MAIN.items():
    slopes = np.array([oc.slope(a) for a in A])
    diff = A[:, 0] - A[:, 4]
    rows.append({"speed": sp, "rm": rm, "negative slope": int((slopes < 0).sum()), "positive bin1-bin5": int((diff > 0).sum()),
                 "sessions": len(A), "slope range": f"{slopes.min():.3f} to {slopes.max():.3f}",
                 "diff range": f"{diff.min():.3f} to {diff.max():.3f}"})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under every reading all 16 sessions have a negative slope of accuracy on
distance bin (-0.072 to -0.007) and a positive bin 1 minus bin 5 difference (0.026 to 0.317).

## fig4e-angle-flat

> Decoding accuracy did not differ between 5 angular sectors around the object, so the distance effect
> is not specific to compass directions.

In [ ]:
rows = []
for sp, rm in READINGS:
    for start in (0.0, 36.0):
        A, _ = run("angle", sp, rm, angle_start=start)
        h, p = kw_table(A)
        rows.append({"speed": sp, "rm": rm, "sectors from": start, "H": h, "p": p,
                     **{f"median {i + 1}": np.median(A[:, i]) for i in range(5)}, "sessions": len(A),
                     "empty sectors": int(np.isnan(A).sum())})
fig4e = pd.DataFrame(rows)
print(fig4e.to_string(index=False))

**Result: blocked (methods underspecified).** The two sector-start readings of the claims file give
different outcomes. With sectors starting at 0°, the Kruskal-Wallis test finds a difference between
sectors under every speed and rate-map reading (H = 9.8 to 13.5, p = 0.0089 to 0.044). With sectors
starting at 36° it does not (H = 6.2 to 8.4, p = 0.077 to 0.19).

## fig4h-population-vector-correlation

> Correlations between NO and O population vectors were lowest near the object and rose stepwise with
> distance before saturating, with significant steps between bins 1 and 2, 2 and 3, and 3 and 4.

Rate maps of the whole NO and O trials, 3 cm bins (50 x 50), smoothed like the decoder's maps (not
blanked), under both speed and rate-map readings. Pearson correlation across cells at each spatial
bin; mean in each distance bin: (a) fixed 20 cm rings from 0 to 100 cm; (b) 5 groups with equal
numbers of spatial bins.

In [ ]:
X3, Y3 = oc.bin_centres(3.0)
rows = []
PV = {}
for sp, rm in READINGS:
    for reading in "ab":
        vals = []
        for s in POP:
            NO, O = s.trials
            mA = oc.rate_maps(*oc.raw_maps(NO, oc.sample_mask(NO, sp == "a"), 3.0), rm=rm, blank=False)
            mB = oc.rate_maps(*oc.raw_maps(O, oc.sample_mask(O, sp == "a"), 3.0), rm=rm, blank=False)
            r = oc.pv_correlation(mA, mB)
            d = np.hypot(X3 - O.obj[0], Y3 - O.obj[1])
            if reading == "a":
                g = np.searchsorted([20, 40, 60, 80, 100], d, side="right")
                g[d >= 100] = -1
            else:
                order = np.argsort(d)
                g = np.empty(len(d), int)
                g[order] = np.minimum(np.arange(len(d)) // (len(d) // 5), 4)
            vals.append([np.nanmean(r[g == i]) for i in range(5)])
        V = np.array(vals)
        PV[sp, rm, reading] = V
        h, p = kw_table(V)
        med = np.median(V, 0)
        pairs = [oc.signrank(V[:, i], V[:, i + 1]) for i in range(4)]
        ok = [pairs[i][1] < 0.05 and med[i + 1] > med[i] for i in range(3)]
        rows.append({"speed": sp, "rm": rm, "bins": reading, "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                     "1v2 p": pairs[0][1], "2v3 p": pairs[1][1], "3v4 p": pairs[2][1], "4v5 p": pairs[3][1],
                     "steps ok": sum(ok), "rising 1->5": med[4] > med[0],
                     "reproduced": p < 0.05 and med[4] > med[0] and all(ok)})
fig4h = pd.DataFrame(rows)
print(fig4h.to_string(index=False))
print("Per speed and rm, any bin reading meeting the criterion:")
print(fig4h.groupby(["speed", "rm"]).reproduced.any().to_string())

**Result: reproduced.** Under every speed and rate-map reading, the fixed 20 cm rings (reading a)
give a Kruskal-Wallis p < 1e-4. Median correlation rises from bin 1 (0.45) to bin 5 (0.77 to 0.80),
with significant rises from bin 1 to 2, 2 to 3 and 3 to 4 (p ≤ 0.013). With equal-count bins
(reading b) the rise from bin 3 to 4 is not significant.

## figs5a-decode-no-from-no-prime

> The NO trial could also be decoded from the NO' trial, above chance, consistent with trace fields.

The population files hold no No Object' trial. The CA1 single-cell file holds it for the same
recordings, with all cells of the two Neuropixels sessions but only 3 to 15 of the 25 to 39 cells of
each tetrode session. The decoder runs on those cells.

In [ ]:
rows = []
for sp, rm in READINGS:
    for k, p in enumerate(POP):
        s = LINK[SESS[k]][0]
        NO, NP = s.trial("NO"), s.trial("NOp")
        pA, pB = oc.half_parts(NO, sp == "a"), oc.half_parts(NP, sp == "a")
        r = oc.decode_pair(pA, pB, "arena", obj=s.trial("O").obj, rm=rm, return_samples=True)
        rng = np.random.default_rng(oc.seed_for("figs5a-decode-no-from-no-prime", sp, rm, k))
        sh = oc.shuffled_accuracy(r["samples"], rng, k=1, n_shuffle=50)
        rows.append({"speed": sp, "rm": rm, "session": SESS[k], "cells": s.n_units, "accuracy": r["acc"][0],
                     "shuffle mean": sh.mean(), "above": r["acc"][0] > 0.5 and r["acc"][0] > sh.mean()})
s5a = pd.DataFrame(rows)
print(s5a.pivot_table(index=["session", "cells"], columns=["speed", "rm"], values="accuracy").to_string())
print(s5a.groupby(["speed", "rm"]).above.agg(["sum", "size"]).to_string())

**Result: reproduced.** NO is told from NO' above 0.5, and above the shuffle mean, in all 16
sessions under every reading: 0.51 to 0.80 in the tetrode sessions, 0.96 to 0.99 in the Neuropixels
sessions. In the tetrode sessions this uses only the 3 to 15 cells of the single-cell file. One
session (26186#2, 3 cells) is at 0.51.

## figs5b-fixed-distance-bins

> With fixed distance bins (0-20 to 80-100 cm) decoding accuracy still fell with distance.

In [ ]:
rows = []
for sp, rm in READINGS:
    A, res = run("fixed", sp, rm, edges_fixed=[0, 20, 40, 60, 80, 100])
    h, p = kw_table(A)
    med = np.nanmedian(A, 0)
    rows.append({"speed": sp, "rm": rm, "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                 "highest": int(np.nanargmax(med)) + 1, "lowest": int(np.nanargmin(med)) + 1,
                 "samples per bin": " ".join(str(int(x)) for x in np.sum([r["n"] for r in res], 0))})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With bins of 0-20 to 80-100 cm, the Kruskal-Wallis test gives H = 51.6 to
53.0 (p < 1e-9) under every reading. The median accuracy is highest in bin 1 (0.81 to 0.82) and
lowest in bin 5 (0.59 to 0.60).

## figs5c-half-trial-control

> Decoding the first from the second half of the NO trial showed no decrease of accuracy with distance.

In [ ]:
rows = []
for sp, rm in READINGS:
    accs = []
    for s in POP:
        NO, O = s.trials
        tm = (NO.t[0] + NO.t[-1]) / 2
        h1, h2 = NO.window(NO.t[0], tm), NO.window(tm, NO.t[-1] + NO.dt)
        r = oc.decode_pair(oc.half_parts(h1, sp == "a"), oc.half_parts(h2, sp == "a"), "distance", obj=O.obj, rm=rm)
        accs.append(r["acc"])
    A = np.array(accs)
    h, p = kw_table(A)
    rows.append({"speed": sp, "rm": rm, "H": h, "p": p, **{f"median {i + 1}": np.median(A[:, i]) for i in range(5)}})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Decoding the first from the second half of the NO trial shows no effect of
distance: Kruskal-Wallis H = 0.16 to 0.67, p ≥ 0.955 under every reading.

## figs5d-control-sessions (blocked)

> In control sessions without an object (CON1 against CON2), decoding accuracy did not fall with
> distance.

The criterion uses the control sessions with at least 25 simultaneously recorded cells. The control
file holds each cell once, and only one of its 14 sessions has 25 or more cells, so a test over
sessions cannot be formed. The claim is blocked. As a variant that does not decide the outcome, the
decoder is run on all 14 control sessions (6 to 30 cells).

In [ ]:
print("Control sessions and cells:", [s.n_units for s in CON1], "; with >= 25 cells:", sum(s.n_units >= 25 for s in CON1))
rows = []
for sp, rm in READINGS:
    for objpos in ((75.0, 75.0), (35.0, 120.0)):
        accs = []
        for s in CON1:
            A_, B_ = s.trial("CON1"), s.trial("CON2")
            r = oc.decode_pair(oc.half_parts(A_, sp == "a"), oc.half_parts(B_, sp == "a"), "distance", obj=objpos, rm=rm)
            accs.append(r["acc"])
        A = np.array(accs)
        h, p = kw_table(A)
        rows.append({"speed": sp, "rm": rm, "object": objpos, "sessions": len(A), "H": h, "p": p,
                     **{f"median {i + 1}": np.median(A[:, i]) for i in range(5)}})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (other).** Only one control session has 25 or more cells. In the variant over all
14 control sessions (6 to 30 cells), the Kruskal-Wallis p is 0.024 to 0.17 across the readings. It
is below 0.05 in two of the eight. This does not decide the outcome.

## figs5e-shuffled-labels

> With shuffled trial labels the decoder performed at 50% in every distance bin.

In [ ]:
rows = []
for (sp, rm), (A, res) in MAIN.items():
    M = []
    for k, r in enumerate(res):
        rng = np.random.default_rng(oc.seed_for("figs5e-shuffled-labels", sp, rm, k))
        M.append(oc.shuffled_accuracy(r["samples"], rng, k=5, n_shuffle=50).mean(0))
    M = np.array(M)
    tt = [stats.ttest_1samp(M[:, i], 0.5) for i in range(5)]
    rows.append({"speed": sp, "rm": rm, **{f"mean {i + 1}": M[:, i].mean() for i in range(5)},
                 **{f"sd {i + 1}": M[:, i].std(ddof=1) for i in range(5)},
                 "min t p": min(t.pvalue for t in tt), "all within 0.49-0.51": bool(np.all(np.abs(M.mean(0) - 0.5) <= 0.01)),
                 "bin1 ok": abs(M[:, 0].mean() - 0.498) <= 0.01, "bin5 ok": abs(M[:, 4].mean() - 0.502) <= 0.01})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked (methods underspecified).** With labels shuffled 50 times per session, the mean
accuracy is within 0.49 to 0.51 in every bin under every reading (0.498 to 0.503). The t tests
against 0.5 give different outcomes by reading. The smallest p is 0.052 (speed a, `rm` a) and 0.22
(speed b, `rm` a), so the criterion holds. It is 0.022 (speed a, `rm` b) and 0.0054 (speed b, `rm` b),
so the criterion fails. Shuffled accuracy need not be exactly 0.5 when a bin's NO and O samples and
the decoder's choices are unbalanced.

## figs5f-random-partition

> Partitioning training and testing samples at random gave the same decline of accuracy with distance.

Reading (a) of the claims file: the trial's 1 s bins permuted at random, the first half training.
Reading (b), permuting position samples, is not run: the claims file does not say how testing
samples would then be formed (asked), and the criterion is met under (a).

In [ ]:
rows = []
for sp, rm in READINGS:
    accs = []
    for k, s in enumerate(POP):
        rng = np.random.default_rng(oc.seed_for("figs5f-random-partition", k))
        NO, O = s.trials
        r = oc.decode_pair(oc.random_parts(NO, sp == "a", rng), oc.random_parts(O, sp == "a", rng), "distance", obj=O.obj, rm=rm)
        accs.append(r["acc"])
    A = np.array(accs)
    h, p = kw_table(A)
    med = np.median(A, 0)
    rows.append({"speed": sp, "rm": rm, "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                 "highest": int(np.argmax(med)) + 1, "lowest": int(np.argmin(med)) + 1})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With 1 s bins partitioned at random, the Kruskal-Wallis test gives H = 26.0
to 30.4 (p < 1e-4) under every reading. The median accuracy is highest in bin 1 (0.84 to 0.85) and
lowest in bin 5 (0.71 to 0.73).

## figs5ghi-fields-and-rates

> Firing fields were spread across the arena, not concentrated at the object, and firing rates were
> similar between NO and O and highest at middle distances (bin 3).

(i) Field centroids in the 2.5 cm maps of NO and O (both `rm` and `field` readings). Per session and
trial: the share of fields with a centroid within 20 cm of the object, divided by the share of the
arena's area within 20 cm of it. Two-sided rank-sum test O against NO over sessions. (ii) Each
cell's mean rate (spikes over time, whole trial) in the 5 equal-sample distance bins of the trial's
position samples, averaged over cells, then over sessions. (iii) Signed-rank test of NO against O in
each bin over sessions.

In [ ]:
X25, Y25 = oc.bin_centres(2.5)
rows = []
for rm in "ab":
    for field in "ab":
        dens = {"NO": [], "O": []}
        for s in POP:
            obj = s.trials[1].obj
            area = np.mean(np.hypot(X25 - obj[0], Y25 - obj[1]) < 20)
            for t in s.trials:
                maps, _ = oc.single_maps(t, rm)
                cents = [oc.field_geometry(idx, maps[u], obj)["d_centroid"] for u in range(s.n_units) for idx in oc.fields(maps[u], 2.5, field)]
                cents = np.array(cents)
                dens[t.name].append(np.mean(cents < 20) / area if len(cents) else np.nan)
        z, p = oc.ranksum(dens["O"], dens["NO"])
        rows.append({"rm": rm, "field": field, "density ratio NO (median)": np.nanmedian(dens["NO"]),
                     "density ratio O (median)": np.nanmedian(dens["O"]), "z (O vs NO)": z, "p": p,
                     "(i) holds": not (p < 0.05 and np.nanmedian(dens["O"]) > np.nanmedian(dens["NO"]))})
part_i = pd.DataFrame(rows)
print(part_i.to_string(index=False))

rates = {"NO": [], "O": []}
for s in POP:
    obj = s.trials[1].obj
    for t in s.trials:
        d = np.hypot(t.x - obj[0], t.y - obj[1])
        e = oc.equal_sample_edges(d)
        g = oc.assign(d, e)
        T = np.bincount(g[g >= 0], minlength=5) * t.dt
        per_cell = []
        for u in range(s.n_units):
            si = t.spike_si[u]
            si = si[si >= 0]
            per_cell.append(np.bincount(g[si][g[si] >= 0], minlength=5) / T)
        rates[t.name].append(np.mean(per_cell, 0))
RN, RO = np.array(rates["NO"]), np.array(rates["O"])
print("\nMean rate per distance bin (Hz), mean over sessions: NO", RN.mean(0).round(3), "O", RO.mean(0).round(3))
print("Peak bin: NO", int(np.argmax(RN.mean(0))) + 1, "O", int(np.argmax(RO.mean(0))) + 1)
sr = [oc.signrank(RN[:, i], RO[:, i]) for i in range(5)]
print("Signed-rank NO vs O per bin, p:", [round(x[1], 4) for x in sr])
pk = (int(np.argmax(RN.mean(0))) + 1, int(np.argmax(RO.mean(0))) + 1)
part_ii = all(k in (2, 3, 4) for k in pk) and 3 in pk
part_iii = all(x[1] >= 0.05 for x in sr)
print("(ii) holds:", part_ii, "; (iii) holds:", part_iii)
print("(i) holds under every rm and field reading:", bool(part_i["(i) holds"].all()))

**Result: not reproduced.** Only one of the three parts holds. (i) holds: fields are not more
concentrated within 20 cm of the object in O than in NO under any reading (rank-sum p = 0.12 to 0.33;
the O ratio is lower). (ii) fails: the mean rate peaks in bin 2 in both trials, not in bin 3 in
either. (iii) fails: NO and O rates differ in bins 3 and 4 (signed-rank p = 0.0027 and 0.011), O
being higher.

## figs6-behaviour-decoders-flat

> Decoders of trial type based on speed, head direction or occupancy performed only slightly above 50%
> and their accuracy did not depend on distance from the object.

Speed (Lowess-smoothed, 0.5 s span, all speeds, gamma likelihood per spatial bin) and occupancy
decoders on 25 Hz samples, with the same halves and the distance-bin edges of the neural decoder's
testing samples. Neuropixels tracking is interpolated to 25 Hz. Bins with fewer than two speed
samples get no gamma fit and are left out of the speed likelihood (a standard choice). The dataset
holds no head direction.

In [ ]:
BEH = {}
rows = []
for kind in ("speed", "occupancy"):
    accs = []
    for k, s in enumerate(POP):
        NO, O = s.trials
        key = (s.index, s.rat)
        if key not in BEH:
            BEH[key] = (oc.Behaviour(NO), oc.Behaviour(O))
        edges = {}
        for ti, tr in enumerate((NO, O)):
            for it, p in enumerate(PARTS[s.index, s.rat, "b"][ti]):
                d = np.hypot(p.pos[:, 0] - O.obj[0], p.pos[:, 1] - O.obj[1])
                edges[ti, it] = oc.equal_sample_edges(d)
        accs.append(oc.behaviour_decoder(*BEH[key], edges, O.obj, kind))
    A = np.array(accs)
    h, p = kw_table(A)
    rows.append({"decoder": kind, "H": h, "p": p, **{f"median {i + 1}": np.median(A[:, i]) for i in range(5)},
                 "median overall": np.median(A)})
s6 = pd.DataFrame(rows)
print(s6.to_string(index=False))
print("Head direction decoder: not run (no head direction in the dataset)")

**Result: not reproduced.** The occupancy decoder's accuracy depends on distance (Kruskal-Wallis
H = 22.8, p = 0.0001; medians 0.56 to 0.71, highest in bin 3). The criterion fails when any of the
three tests gives p < 0.05. The speed decoder shows no effect of distance (H = 8.9, p = 0.062;
medians 0.52 to 0.56). The head-direction decoder cannot be run, as the dataset holds no head
direction, but that does not change the outcome. Median accuracy is 0.54 (speed) and 0.62
(occupancy).

## figs6hij-behaviour-matched

> After matching the distributions of speed, head direction or occupancy between NO and O within
> each distance bin, decoding accuracy on neural data still fell with distance.

Matching as in the Methods (speed bins of 5 cm/s to 100 cm/s, occupancy bins of 30 cm), random
removal of the excess testing samples of the trial with more, within each iteration and distance
bin. Readings of the claims file: (1) the 1 s testing samples matched, with behaviour averaged over
the second; (2) 25 Hz samples matched, keeping the 1 s bins all of whose samples survive. The
decisions are those of the main decoder.

In [ ]:
def match_keep(groupsA, groupsB, rng):
    """Indices kept in A and B so that their counts match in every group."""
    keepA, keepB = [], []
    for g in np.union1d(groupsA[groupsA >= 0], groupsB[groupsB >= 0]):
        ia, ib = np.flatnonzero(groupsA == g), np.flatnonzero(groupsB == g)
        n = min(len(ia), len(ib))
        keepA.append(rng.choice(ia, n, replace=False))
        keepB.append(rng.choice(ib, n, replace=False))
    return (np.concatenate(keepA) if keepA else np.zeros(0, int)), (np.concatenate(keepB) if keepB else np.zeros(0, int))


def behaviour_group(kind, speed, x, y):
    if kind == "speed":
        g = np.floor(speed / 5).astype(int)
        g[(speed < 0) | (speed >= 100)] = -1
        return g
    return np.clip(np.floor(x / 30), 0, 4).astype(int) * 5 + np.clip(np.floor(y / 30), 0, 4).astype(int)


rows = []
for (sp, rm), (A0, res) in MAIN.items():
    for kind in ("speed", "occupancy"):
        for reading in ("1", "2"):
            accs = []
            for k, (s, r) in enumerate(zip(POP, res)):
                rng = np.random.default_rng(oc.seed_for("figs6hij-behaviour-matched", sp, rm, kind, reading, k))
                bA, bB = BEH[s.index, s.rat]
                corr, tot = np.zeros(5), np.zeros(5)
                for it in range(2):
                    S = [x for x in r["samples"] if x["iteration"] == it]
                    per = []
                    for x in S:
                        p = PARTS[s.index, s.rat, sp][x["label"]][it]
                        b = (bA, bB)[x["label"]]
                        st, en = p.starts[x["index"]], p.ends[x["index"]]
                        lo, hi = np.searchsorted(b.t, st), np.searchsorted(b.t, en)
                        if reading == "1":
                            sp_mean = np.array([b.speed[a:c].mean() if c > a else np.nan for a, c in zip(lo, hi)])
                            xm = np.array([b.x[a:c].mean() if c > a else np.nan for a, c in zip(lo, hi)])
                            ym = np.array([b.y[a:c].mean() if c > a else np.nan for a, c in zip(lo, hi)])
                            g = behaviour_group(kind, np.nan_to_num(sp_mean, nan=-1), np.nan_to_num(xm), np.nan_to_num(ym))
                            g[~np.isfinite(sp_mean)] = -1
                            per.append((x, g * 5 + x["ring"]))
                        else:
                            per.append((x, lo, hi))
                    if reading == "1":
                        (xa, ga), (xb, gb) = per
                        ka, kb = match_keep(np.where(ga >= 0, ga, -1), np.where(gb >= 0, gb, -1), rng)
                        for xx, kk, lab in ((xa, ka, 0), (xb, kb, 1)):
                            dec, ring = xx["decision"][kk], xx["ring"][kk]
                            v = dec >= 0
                            np.add.at(tot, ring[v], 1)
                            np.add.at(corr, ring[v], dec[v] == lab)
                    else:
                        # 25 Hz samples of the testing windows, grouped by behaviour and the window's ring
                        sam = []
                        for (xx, lo, hi), b in zip(per, (bA, bB)):
                            idx = np.concatenate([np.arange(a, c) for a, c in zip(lo, hi)]) if len(lo) else np.zeros(0, int)
                            win = np.repeat(np.arange(len(lo)), hi - lo)
                            g = behaviour_group(kind, b.speed[idx], b.x[idx], b.y[idx])
                            g = np.where(g >= 0, g * 5 + xx["ring"][win], -1)
                            sam.append((idx, win, g))
                        ka, kb = match_keep(sam[0][2], sam[1][2], rng)
                        for (xx, lo, hi), (idx, win, g), kk, lab in zip(per, sam, (ka, kb), (0, 1)):
                            survived = np.zeros(len(idx), bool)
                            survived[kk] = True
                            full = np.bincount(win, weights=survived, minlength=len(lo)) == (hi - lo)
                            full &= (hi - lo) > 0
                            dec, ring = xx["decision"][full], xx["ring"][full]
                            v = dec >= 0
                            np.add.at(tot, ring[v], 1)
                            np.add.at(corr, ring[v], dec[v] == lab)
                accs.append(corr / tot)
            A = np.array(accs)
            h, p = kw_table(A)
            med = np.nanmedian(A, 0)
            rows.append({"speed": sp, "rm": rm, "matched": kind, "reading": reading, "H": h, "p": p,
                         **{f"median {i + 1}": med[i] for i in range(5)}, "bin 1 highest": int(np.nanargmax(med)) == 0})
s6m = pd.DataFrame(rows)
print(s6m.to_string(index=False))
print(f"({time.time() - started:.0f} s so far)")

**Result: blocked (data missing).** After matching speed or occupancy, the neural decoder's accuracy
still falls with distance under every reading and both matching readings: Kruskal-Wallis H = 27.3 to
50.5, p < 1e-4, bin 1 highest. The head-direction match, one of the three the criterion needs,
cannot be made, as the dataset holds no head direction.

## figs6k-uniform-prior

> Replacing the occupancy prior by a uniform prior did not change the decline of decoding accuracy with
> distance.

In [ ]:
rows = []
for sp, rm in READINGS:
    A, _ = run("distance", sp, rm, uniform=True)
    h, p = kw_table(A)
    med = np.median(A, 0)
    rows.append({"speed": sp, "rm": rm, "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                 "bin 1 highest": int(np.argmax(med)) == 0})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** With a uniform prior the Kruskal-Wallis test gives H = 46.9 to 48.6 (p < 1e-8),
with bin 1 highest, under every reading (paper H = 42.92).

## figs6l-sampling-thresholds

> Using only spatial bins visited for at least 100, 500, 1000 or 2000 ms in both trials, the decline of
> accuracy with distance remained.

10 cm rate-map bins. The claims file does not say whether a bin's time is that of the whole trial
or of the training half, nor whether testing samples in left-out bins are kept (asked); all four
combinations are run: occupancy (w) whole trial or (t) training half; testing samples (k) kept or (d)
dropped.

In [ ]:
B10 = oc.nbins(10.0) ** 2
rows = []
for sp, rm in READINGS:
    for occ_reading in "wt":
        for samp in "kd":
            for thr in (0.1, 0.5, 1.0, 2.0):
                accs = []
                for s in POP:
                    NO, O = s.trials
                    pA, pB = PARTS[s.index, s.rat, sp]
                    if occ_reading == "w":
                        occs = [np.bincount(oc.bin_index(t.x, t.y, 10.0), minlength=B10) * t.dt for t in (NO, O)]
                        keep = (occs[0] >= thr) & (occs[1] >= thr)
                    else:
                        keep = []
                        for it in range(2):
                            occs = [np.bincount(oc.bin_index(p.trial.x[p.train_mask], p.trial.y[p.train_mask], 10.0), minlength=B10) * p.trial.dt
                                    for p in (pA[it], pB[it])]
                            keep.append((occs[0] >= thr) & (occs[1] >= thr))
                    filt = None
                    if samp == "d":
                        def filt(p, label, ring, keep=keep):
                            kb = keep if not isinstance(keep, list) else keep[[pA, pB][label].index(p)]
                            return kb[oc.bin_index(p.pos[:, 0], p.pos[:, 1], 10.0)]
                    r = oc.decode_pair(pA, pB, "distance", obj=O.obj, rm=rm, bin_cm=10.0, keep_bins=keep, sample_filter=filt,
                                       cache=CACHE.setdefault((s.index, s.rat, sp), {}))
                    accs.append(r["acc"])
                A = np.array(accs)
                h, p = kw_table(A)
                med = np.nanmedian(A, 0)
                rows.append({"speed": sp, "rm": rm, "occupancy": occ_reading, "samples": samp, "threshold s": thr,
                             "H": h, "p": p, **{f"median {i + 1}": med[i] for i in range(5)},
                             "bin 1 highest": int(np.nanargmax(med)) == 0})
s6l = pd.DataFrame(rows)
print(s6l.to_string(index=False))
s6l["ok"] = (s6l.p < 0.05) & s6l["bin 1 highest"]
print("\nAll four thresholds meet the criterion, per reading:")
print(s6l.groupby(["speed", "rm", "occupancy", "samples"]).ok.all().to_string())

**Result: reproduced.** Under every combination of readings, all four thresholds give a
Kruskal-Wallis p < 1e-5 (H = 32.6 to 42.8), with bin 1 highest.

## tables1-without-bin-1

> The effect of distance on decoding accuracy held when the bin nearest the object was left out.

In [ ]:
rows = []
for (sp, rm), (A, _) in MAIN.items():
    h, p = kw_table(A[:, 1:])
    med = np.median(A[:, 1:], 0)
    rows.append({"speed": sp, "rm": rm, "H bins 2-5": h, "p": p, "bin 2 highest": int(np.argmax(med)) == 0,
                 "H within 25.06+/-2.506": abs(h - 25.06) <= 2.506})
print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Over bins 2 to 5 the Kruskal-Wallis test gives p < 1e-5 with bin 2 highest,
under every reading. H is 28.4 to 32.8, above 25.06 ± 2.506 under every reading: the effect is
stronger here than in the paper.

## figs5jk-walls-no-pivot

> The arena walls did not act as a pivot as the object did; position decoding error and population
> vector correlations were comparable near (under 15 cm) and far from the walls.

(J) Position decoded within the NO trial (train on one half, test on the other), error of (a) the
maximum-posterior bin and (b) the posterior mean; per session the mean error near and far from a
wall. (K) Population vector correlation between the maps of the first and second halves of NO,
mean over the spatial bins near and far. Signed-rank tests over the 16 sessions.

In [ ]:
rows = []
for sp, rm in READINGS:
    near_far = {"a": [], "b": []}
    pvnf = []
    for s in POP:
        NO = s.trials[0]
        pos, e_map, e_mean = oc.position_errors(PARTS[s.index, s.rat, sp][0], rm)
        wall = np.min(np.c_[pos, 150 - pos], axis=1)
        for key, e in (("a", e_map), ("b", e_mean)):
            near_far[key].append((e[wall < 15].mean(), e[wall >= 15].mean()))
        tm = (NO.t[0] + NO.t[-1]) / 2
        m1 = oc.rate_maps(*oc.raw_maps(NO, oc.sample_mask(NO, sp == "a", None, tm), 3.0), rm=rm, blank=False)
        m2 = oc.rate_maps(*oc.raw_maps(NO, oc.sample_mask(NO, sp == "a", tm, None), 3.0), rm=rm, blank=False)
        r = oc.pv_correlation(m1, m2)
        wb = np.min(np.c_[X3, Y3, 150 - X3, 150 - Y3], axis=1)
        pvnf.append((np.nanmean(r[wb < 15]), np.nanmean(r[wb >= 15])))
    for key in "ab":
        nf = np.array(near_far[key])
        w, p = oc.signrank(nf[:, 0], nf[:, 1])
        rows.append({"speed": sp, "rm": rm, "measure": f"error ({'MAP' if key == 'a' else 'posterior mean'})",
                     "near median": np.median(nf[:, 0]), "far median": np.median(nf[:, 1]),
                     "median difference": np.median(nf[:, 0] - nf[:, 1]), "W": w, "p": p})
    nf = np.array(pvnf)
    w, p = oc.signrank(nf[:, 0], nf[:, 1])
    rows.append({"speed": sp, "rm": rm, "measure": "PV correlation halves", "near median": np.median(nf[:, 0]),
                 "far median": np.median(nf[:, 1]), "median difference": np.median(nf[:, 0] - nf[:, 1]), "W": w, "p": p})
s5jk = pd.DataFrame(rows)
print(s5jk.to_string(index=False))
print("Tests with p < 0.05:", int((s5jk.p < 0.05).sum()), "of", len(s5jk))

**Result: not reproduced.** All 12 signed-rank tests (4 readings x 3 measures) find a difference
between near and far from the walls (p = 0.0001 to 0.013). Position errors are 7.4 to 10.2 cm larger
near the walls (medians 35 to 39 against 27 to 29 cm). Population vector correlations between the
two halves of NO are 0.05 to 0.08 lower near the walls.

## figs5m-without-object-tuned

> After removing all object-tuned cells, decoding accuracy still fell gradually with distance.

The object-tuned cells are those of the template matching of fig1c (notebook 01, recomputed here
with the same seeds) among the 897 population cells linked to the CA1 file. The other 283 tetrode
cells have no Object Moved trial in the dataset and so no tuning status. Two variants: (r1) the
linked tuned cells removed; (r2) also every unlinked cell removed.

In [ ]:
rows = []
for rmt in "ab":
    for w in "ab":
        tun = oc.object_tuning(CA1, rmt, oc.WIDTHS[w], key="CA1")
        tuned = {(c["session"], c["unit"]) for c in tun if c["tuned"]}
        for variant in ("r1", "r2"):
            def keep_units(s, variant=variant, tuned=tuned):
                cs, m = LINK[f"{s.rat}#{s.index}"]
                keep = [u for u in range(s.n_units) if (u in m and (cs.index, m[u]) not in tuned) or (u not in m and variant == "r1")]
                return keep
            for sp in "ab":
                A, _ = run("distance", sp, rmt, units=keep_units)
                h, p = kw_table(A)
                med = np.median(A, 0)
                removed = sum(s.n_units - len(keep_units(s)) for s in POP)
                rows.append({"rm": rmt, "w": w, "variant": variant, "speed": sp, "cells removed": removed, "H": h, "p": p,
                             **{f"median {i + 1}": med[i] for i in range(5)}, "bin 1 highest": int(np.argmax(med)) == 0})
s5m = pd.DataFrame(rows)
print(s5m.to_string(index=False))
print(f"\nTotal time {time.time() - started:.0f} s")

**Result: blocked (methods underspecified).** It depends on fig1c-object-tuned-fraction, which is blocked (methods underspecified), so it is blocked for the same reason (protocol, Author 1.5); the result under every reading is given here. Without the object-tuned cells among the linked cells (variant r1, 181 to 272
cells removed), the Kruskal-Wallis test gives H = 43.8 to 46.8, p < 1e-8, with bin 1 highest, under
every template-width, rate-map and speed reading. The 283 tetrode cells of unknown status stay in.
Removing them too (r2, 464 to 555 cells removed) leaves bin 1 highest, with p = 0.026 to 0.050 under
`rm` a and 0.071 to 0.17 under `rm` b.

## Summary

| Claim | Outcome |
|---|---|
| fig3b-decode-presence | reproduced |
| fig3c-bin-size | reproduced |
| fig4b-distance-decline | reproduced |
| fig4c-each-session-decreasing | reproduced |
| fig4e-angle-flat | blocked, methods underspecified (sector-start readings differ) |
| fig4h-population-vector-correlation | reproduced |
| figs5a-decode-no-from-no-prime | reproduced |
| figs5b-fixed-distance-bins | reproduced |
| figs5c-half-trial-control | reproduced |
| figs5d-control-sessions | blocked, other (one control session with 25 or more cells) |
| figs5e-shuffled-labels | blocked, methods underspecified (t tests differ by reading) |
| figs5f-random-partition | reproduced |
| figs5ghi-fields-and-rates | not reproduced (one of three parts) |
| figs6-behaviour-decoders-flat | not reproduced (occupancy decoder depends on distance) |
| figs6hij-behaviour-matched | blocked, data missing (head direction) |
| figs6k-uniform-prior | reproduced |
| figs6l-sampling-thresholds | reproduced |
| tables1-without-bin-1 | reproduced (H above the paper's) |
| figs5jk-walls-no-pivot | not reproduced |
| figs5m-without-object-tuned | blocked, depends on fig1c (reproduced under every reading of variant r1) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the decoding sessions are the 16 of the population
files, with the Neuropixels sessions' tracking taken from the CA1 file; the trials are shorter than
the 1800 s the Methods assume; the No Object' decoding of the tetrode sessions uses the cells of the
single-cell file only; Neuropixels tracking is interpolated to 25 Hz for the behaviour decoders;
head direction is not available; object-tuned cells can be removed only among the 897 cells linked
to the CA1 file.

What was found about the dataset: the y coordinate of the tracking in the two Neuropixels
population files is a copy of x; the population files hold the No Object and Object trials only;
head direction is not deposited; the control file has one session with 25 or more cells.